# Labeling
We label the data collected in [01_data_collection.ipynb](01_data_collection.ipynb) using zero-shot classification via `facebook/bart-large-mnli`.

In [1]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Next, we define some `relevant` categories and some `irrelevant` categories.

In [2]:
relevant = [
    "Monetary and fiscal policy",
    "Macroeconomic data and indicators",
    "Geopolitics",
    "International trade",
    "Corporate and sector dynamics",
    "Commodities and supply Chain",
    "Systemic shocks",
    "Natural environment shocks",
    "Public health",
    "Technology advancements",
]

irrelevant = [
    "Entertainment",
    "Culture and art",
    "Sports",
    "Lifestyle, travel, wellness",
    # "Local news",
    "Science and pure research",
    "Long-form Features & Investigative Essays",
    "Personal & Human Interest",
]

We use `HFGoodBadLabeler` to label sequences according to the given relevant/irrelevant classes.  We also define a `score_jittery` function to compute a jittery score.

In [3]:
from labeling import HFGoodBadLabeler

label_relevant = HFGoodBadLabeler(classifier, relevant, irrelevant, high=0.65, low=0.35)

score_jittery = lambda sequence: classifier(sequence, "jittery")["scores"][0]

Here is an example.

In [4]:
sentence = "US military base in Germany to close after 100 years of activity"

print("Relevant:", label_relevant(sentence))
print("Score:", score_jittery(sentence))

Relevant: -1
Score: 0.6441225409507751


### Working on the collected dataset
We now load, clean, and label the `raw_headlines.csv` dataset.

We first merge title and description into a HuggingFace `Dataset` for faster processing.

In [5]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("data/raw_headlines.csv")
df["concat"] = df["title"] + " | " + df["description"]
ds = Dataset.from_pandas(df[["concat"]].dropna(inplace=False))

Then we define the batch processing function.

In [6]:
def process_batch(batch):
    inputs = batch["concat"]

    relevance = [label_relevant(x) for x in inputs]
    score = [score_jittery(x) for x in inputs]

    return {"relevance": relevance, "score": score}

In [7]:
len(ds["concat"])

1164

We apply the processing function to the dataset

In [9]:
results = ds.map(process_batch, batched=True)

Map:   0%|          | 0/1164 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


and copy the new columns in the original `pandas` dataframe:

In [12]:
df.dropna(inplace=True)
df["relevance"] = results["relevance"]
df["score"] = results["score"]

In [19]:
df.to_csv("data/labeled_headlines.csv")